# PT 2: 3D Launch Geometry and Driver Pipeline

The goal of this notebook is to unpack how our 4D tensors are mapped to hardware threads on the GPU. How launch geometry is calculated and cached, and how drivers manage memory pass throughs. We've already shown how much faster the custom kernels are, but we still have to peel back all the layers to understand why it works so fast. 
If you have been reading along and haven't already, I recommend still skimming through the two files, that way it's a little easier to follow along in the notebooks. 

<img src="../../../notebooks/assets/max_pooling_cfd.png" width="1200" alt="Max Pooling CFD Flowchart">


### Mapping our Tensors to 3D Thread Grids via (`_get_launch_geometry`): 

The flowchart is a great reminder in knowing how `Layer._compile_for_device` works. In this case it's for max pooling as its slightly more complicated then `AvgPool2d`. 


Now, we'd like to explore what happens when we take the forward and backward route of the gpu layers. So lets dive in!
While its a bit wiser to explain how we map the kernels, I'd rather wait until later on in the notebook, so we'll discuss the main helper methods inside class `_PoolNd` first. 

Our normal `self.forward` layer is replaced with one of the two dynamic methods at runtime.

In [2]:
import numpy as np
class MaxPool2d_ex(): # base class Layer omitted
    def _forward_gpu(self, inputs, training, block_z):
        kernel = self._kernel_train if training else self._kernel_infer
        return self._forward_gpu_common(
            inputs,
            pad_value=-np.inf,
            block_z=block_z,
            kernel=kernel,
            track_indices=training,
        )

    def _backward_gpu(self, dvalues, block_z):
        return self._backward_gpu_common(
            dvalues,
            block_z,
            kernel=self._kernel_backward_nonoverlap,
            aux_args=(self.max_indices,),
        )

The forward and backward passes both pass in similar arguments, lets start with the forward pass: 
* `inputs`: Our input tensor we perform the calculation on.
* `pad_value`: we hardcode `-np.inf` as noted in the 1st notebook, average pooling would require `0.0`. 
* `block_z`: Either a block depth of 2 or 4 depending on backend type
* `kernel`: Our custom kernel that is compiled once
* `track_indices`: Given the training flag (will likely be deprected) if true will keep a 4D tensor that tracks the `self.max_indices` for backpropagation. 

The backward pass will pass in: 
`aux_args`: This is needed to compute the location of the winners from the forward pass, then we'll route 100% of the gradient directly to the max positon. 

Both average and max pooling classes will then go inside `_PoolNd._forward_gpu_common`. This layer will handle the forward input `inputs.padded`, setup launch geometry `_PoolNd._get_launch_geometry`, handle the max vs inference logic for max pooling, and call our custom kernel. The first notebook already went over `self._prepare_forward_input` so we'll go inside `_PoolNd._get_launch_geometry` instead. 

In [ ]:
class _PoolNd():
    def __init__(self):
        self._launch_cache = {}

    def _get_launch_geometry(self, S, H_pad, W_pad, C, H_out, W_out, block_z):

        key = (S, H_pad, W_pad, C, H_out, W_out, block_z)
        cached = self._launch_cache.get(key)
        if cached is not None:
            return cached

        fH, fW = self.filter_size
        sH, sW = self.stride

        block_x = min(32, C)
        block_y = 8
        block_dim = (block_x, block_y, block_z)
 
        grid_x = (C + block_x - 1) // block_x
        grid_y = (W_out + block_y - 1) // block_y
        grid_z = (H_out * S + block_z - 1) // block_z
        grid_dim = (grid_x, grid_y, grid_z)
 
        static_args = (
            np.int32(S), np.int32(H_pad), np.int32(W_pad), np.int32(C),
            np.int32(fH), np.int32(fW), np.int32(sH), np.int32(sW),
            np.int32(H_out), np.int32(W_out),
        )
 
        result = (block_dim, grid_dim, static_args)
        self._launch_cache[key] = result
        return result
    
    def _forward_gpu_common(self, inputs, pad_value, block_z, kernel, track_indices=False):
        xp, inputs_padded, S, H_out, W_out = self._prepare_forward_input(inputs, pad_value=pad_value)

        self.inputs_shape = inputs.shape
        self.padded_shape = inputs_padded.shape
        _, H_pad, W_pad, C = inputs_padded.shape

        self.output = xp.empty((S, H_out, W_out, C), dtype=inputs.dtype)
        block_dim, grid_dim, static_args = self._get_launch_geometry(
            S, H_pad, W_pad, C, H_out, W_out, block_z
        )

        if track_indices:
            self.max_indices = xp.empty((S, H_out, W_out, C), dtype=xp.int32)
            aux_args = (self.max_indices,)
        else:
            self.max_indices = None
            aux_args = ()

        kernel_args = (inputs_padded, self.output) + aux_args + static_args
        kernel(grid_dim, block_dim, kernel_args)

        return self.output

### How does `self._get_launch_geometry` work and why is it a $O(1)$ Lookup? 

We'll use a dictionary for this part, our key becomes the input parameters, and we'll check if we already cached the result. If not, we'll setup the actual grid dimensions, blocks, and static arguments needed for our kernel. These are needed for when we call the kernel later. 
```python
key = (S, H_pad, W_pad, C, H_out, W_out, block_z)
cached = self._launch_cache.get(key)
if cached is not None:
    return cached #this contains the tuple (block_dim, grid_dim, static_args)
# Used in static_args
fH, fW = self.filter_size
sH, sW = self.stride

block_x = min(32, C)
block_y = 8
#block_z = 2 or 4
block_dim = (block_x, block_y, block_z)
```

As a quick reminder, memory transactions are done usually through 32 threads per transaction, meaning we'll conform our block size to be divisible by 32 (512 for HIP, 1024 for CUDA). 

Hardware & Execution Context (GPU Architecture Alignment)
* `block_x` **& Warp Coalescing**: Setting block_x = min(32, C) directly maps the inner dimension to a single 32-thread hardware Warp (NVIDIA) or Wavefront (AMD). When $C \ge 32$, all 32 threads in the warp execute together in lockstep, merging 32 individual memory accesses into a single 128-byte coalesced memory transaction over the VRAM bus.

* `block_z` **& L1 Cache / Occupancy Tuning**: block_z assigns 3D depth (channels/batches) to adjacent threads within the same physical compute core (SM or CU).
    * L1 Cache Reuse: Neighboring $Z$-threads process adjacent channels/batches simultaneously, allowing them to hit the same local L1 cache lines with minimal latency.

* **Latency Hiding via SM Warp Scheduling**:

Memory transactions from VRAM to cache take **200-400** clock cycles (modern consumer GPUs sit around 2.7-3.2 GHz). GPUs hide this latency by maintaining thousands of active thread states directly on-chip and switching execution instantly when a memory fetch stalls a thread group.

* **NVIDIA (Streaming Multiprocessor - SM)**: 
    * **State Allocation**: Up to **64 active Warps (2,048 threads)** stay resident in the SM's **64K 32-bit Register File.**
    * **Scheduling**: Four independent **Warp Schedulers** check resident warps every clock cycle. When Warp 0 stalls on a VRAM fetch, the schedular instantly swaps executions to an eligible Warp on the next clock edge with **0-cycle overhead**. 
* **AMD (RDNA Archtiecture - CU/WGP)**:
    * **State Allocation**: Up to **40 Wavefronts** stay resident per Compute Unit (CU) allocated across the physical **Vector General Purpose Register (VGPR)** pool 
    * **Four Wavefront Schedulers per CU** manage SIMD execution vector units. If Wavefront 0 stalls on a global BUFFER_LOAD, the hardware scheduler immediately hands the SIMD unit to Wavefront 1 with **0-cycle latency**, leveraging on-chip VGPR state retention.
    > **Architectural Note (RDNA 4 Dynamic VGPRs) [^1]:** > Modern AMD architectures (RDNA 4+) introduce *Dynamic VGPR Allocation*. Rather than statically reserving peak register space for a Wavefront's full lifecycle, the hardware dynamically allocates and frees 1024-bit vector registers on the fly based on runtime demands. This drastically lowers register pressure (a power user can now set block_z = 4), allowing higher resident Wavefront occupancy on the CU for improved latency hiding.

    [^1]: Chips and Cheese: [Dynamic Register Allocation on AMD's RDNA 4 Architecture](https://chipsandcheese.com/p/dynamic-register-allocation-on-amds)

At a higher level, our $Z$-axis enables our latency hiding (as shown above), avoiding the VRAM transaction and means that we can perform the pooling kernels on multiple threads. 

### HOW DOES THE GRID... GRID???

Right after we define how threads form the Warp or Wavefront inside a single thread block, we have to observe how thread blocks are distributed across the entire dataset using a Grid. 
```python
grid_x = (C + block_x - 1) // block_x
grid_y = (W_out + block_y - 1) // block_y
grid_z = (H_out * S + block_z - 1) // block_z
grid_dim = (grid_x, grid_y, grid_z)
```
$$
\text{Ceiling Division} \lceil N/\text{block}\rceil = (N + \text{block}-1) // \text{block}
$$
This formula is standard GPU hardware arithmetic for ceiling division. We can't make "half blocks" during our run, so we'll force wasted threads if some are left empty. 

GRID DIMENSIONS **(grid_x, grid_y, grid_z)**  
  grid_x ───────┤ Channels (C)                            
  grid_y ───────┤ Output Width (W_out)                    
  grid_z ───────┤ Output Height × Batch (H_out × S)       
